In [5]:
import base64
import datetime
import json
import asyncio
import websockets
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.backends import default_backend

API_KEY = '982c924c-9e22-44c8-a801-3be1ff50d45d'
KEY_FILE = 'Key1.txt'
WS_URL  = 'wss://external-api-ws.kalshi.com/trade-api/ws/v2'

In [6]:
def load_private_key(file_path):
    with open(file_path, 'rb') as f:
        return serialization.load_pem_private_key(f.read(), password=None, backend=default_backend())

def make_auth_headers(private_key):
    ts_ms = str(int(datetime.datetime.now().timestamp() * 1000))
    msg = ts_ms + 'GET' + '/trade-api/ws/v2'
    sig = base64.b64encode(
        private_key.sign(
            msg.encode('utf-8'),
            padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.DIGEST_LENGTH),
            hashes.SHA256()
        )
    ).decode('utf-8')
    return {
        'KALSHI-ACCESS-KEY': API_KEY,
        'KALSHI-ACCESS-SIGNATURE': sig,
        'KALSHI-ACCESS-TIMESTAMP': ts_ms,
    }

In [4]:
# Set MARKET_TICKER to any active Kalshi market ticker you want to watch.
# Example ATP match ticker from kalshi_live_data.ipynb:
MARKET_TICKER = 'KXATPCHALLENGERMATCH-26JUN27BALNAG-BAL'
N_MESSAGES = 10000  # stop after receiving this many messages

async def stream(market_ticker, n_messages):
    private_key = load_private_key(KEY_FILE)
    headers = make_auth_headers(private_key)

    async with websockets.connect(WS_URL, additional_headers=headers) as ws:
        print(f'Connected to {WS_URL}')

        subscribe = {
            'id': 1,
            'cmd': 'subscribe',
            'params': {
                'channels': ['ticker'],
                'market_tickers': [market_ticker],
            }
        }
        await ws.send(json.dumps(subscribe))
        print(f'Subscribed to ticker channel for {market_ticker}')

        for _ in range(n_messages):
            msg = json.loads(await ws.recv())
            print(json.dumps(msg, indent=2))

await stream(MARKET_TICKER, N_MESSAGES)

Connected to wss://external-api-ws.kalshi.com/trade-api/ws/v2
Subscribed to ticker channel for KXATPCHALLENGERMATCH-26JUN27BALNAG-BAL
{
  "type": "subscribed",
  "id": 1,
  "msg": {
    "channel": "ticker",
    "sid": 1
  }
}
{
  "type": "ticker",
  "sid": 1,
  "msg": {
    "market_id": "9c8e9273-f071-42ca-9c06-68deb6a33543",
    "market_ticker": "KXATPCHALLENGERMATCH-26JUN27BALNAG-BAL",
    "price_dollars": "0.2800",
    "yes_bid_dollars": "0.2800",
    "yes_ask_dollars": "0.2900",
    "volume_fp": "2113543.12",
    "open_interest_fp": "1452689.28",
    "dollar_volume": 1056771,
    "dollar_open_interest": 726344,
    "yes_bid_size_fp": "6436.44",
    "yes_ask_size_fp": "33613.55",
    "last_trade_size_fp": "11.16",
    "ts": 1782573631,
    "ts_ms": 1782573631453,
    "time": "2026-06-27T15:20:31.453606Z"
  }
}
{
  "type": "ticker",
  "sid": 1,
  "msg": {
    "market_id": "9c8e9273-f071-42ca-9c06-68deb6a33543",
    "market_ticker": "KXATPCHALLENGERMATCH-26JUN27BALNAG-BAL",
    "price

CancelledError: 

In [ ]:
import requests
import time

MILESTONE_ID = 'bd6fd6f0-a15e-4e99-b4a8-82b612ad79f0'
POLL_INTERVAL = 5  # seconds

url = f'https://external-api.kalshi.com/trade-api/v2/live_data/milestone/{MILESTONE_ID}'

print(f'Polling {url} every {POLL_INTERVAL}s. Interrupt kernel to stop.\n')
while True:
    r = requests.get(url)
    print(f'[{datetime.datetime.now().strftime("%H:%M:%S")}] {r.status_code}')
    print(json.dumps(r.json(), indent=2))
    print()
    time.sleep(POLL_INTERVAL)

In [7]:
MILESTONE_ID = '5b341ea0-5357-48a9-9c29-85325f0fba95'
WS_MILESTONE_URL = f'wss://external-api-ws.kalshi.com/trade-api/v2/live_data/milestone/{MILESTONE_ID}'

async def stream_milestone():
    private_key = load_private_key(KEY_FILE)

    # Sign against the milestone WS path (same pattern as the ticker WS)
    ts_ms = str(int(datetime.datetime.now().timestamp() * 1000))
    path = f'/trade-api/v2/live_data/milestone/{MILESTONE_ID}'
    sig = base64.b64encode(
        private_key.sign(
            (ts_ms + 'GET' + path).encode('utf-8'),
            padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.DIGEST_LENGTH),
            hashes.SHA256()
        )
    ).decode('utf-8')
    headers = {
        'KALSHI-ACCESS-KEY': API_KEY,
        'KALSHI-ACCESS-SIGNATURE': sig,
        'KALSHI-ACCESS-TIMESTAMP': ts_ms,
    }

    print(f'Connecting to {WS_MILESTONE_URL}')
    async with websockets.connect(WS_MILESTONE_URL, additional_headers=headers) as ws:
        print('Connected. Waiting for messages...')
        async for raw in ws:
            print(json.dumps(json.loads(raw), indent=2))

await stream_milestone()

Connecting to wss://external-api-ws.kalshi.com/trade-api/v2/live_data/milestone/5b341ea0-5357-48a9-9c29-85325f0fba95


InvalidStatus: server rejected WebSocket connection: HTTP 404